# Day 2 Instructor Demo: vectors and similarity (D2.1–D2.4)

**Run by the instructor only** (Slides 7–9); shared with participants afterwards. Needs the same Secrets as the
code-along. ⚠️ Never put company data into free APIs.

**Before the session:** run D2.3 and D2.4 once and write the real scores on a sticky note.

In [ ]:
# Setup: downloads llm_client.py and connects to your provider
import importlib, os, shutil, urllib.request
from pathlib import Path
REPO = "https://raw.githubusercontent.com/mari-muthu-k/agent-to-production/main"
FILES = {'llm_client.py': 'paper-summarizer/paper_agent/llm_client.py', 'tinycoder.pdf': 'notebooks/day2/tinycoder.pdf', 'fake.pdf': 'notebooks/day2/fake.pdf', 'sixty_pages.pdf': 'notebooks/day2/sixty_pages.pdf', 'tinycoder_embeddings.gemini-embedding-001.json': 'notebooks/day2/tinycoder_embeddings.gemini-embedding-001.json', 'tinycoder_embeddings.mock-embed.json': 'notebooks/day2/tinycoder_embeddings.mock-embed.json'}

def fetch(name, force=False):
    """Copy a course file next to the notebook: from the repo checkout (Docker), else from GitHub (Colab)."""
    if Path(name).exists() and not force:
        return
    local = Path(os.environ.get("COURSE_REPO", "/nonexistent")) / FILES[name]
    if local.exists():
        shutil.copy(local, name)
    else:
        urllib.request.urlretrieve(f"{REPO}/{FILES[name]}", name)

fetch("llm_client.py")

# Your provider, from Colab Secrets (🔑, notebook access ON) or environment variables. Table at the top.
# ⚠️ Never put company data into free APIs.
import json, logging, re, hashlib, base64
from typing import Optional
import numpy as np
NAMES = ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL", "EMBED_MODEL", "LLM_FALLBACK_MODEL", "OTHER_EMBED_MODEL",
         "LLM_REASONING_EFFORT")
try:
    from google.colab import userdata
    for name in NAMES:
        try:
            if userdata.get(name):
                os.environ[name] = userdata.get(name)
        except Exception:                      # secret not defined, or notebook access is off
            pass
except ImportError:
    pass
missing = [n for n in NAMES[:4] if not os.environ.get(n)]
if missing:
    raise RuntimeError(f"Missing {missing}: add them in Colab Secrets (🔑) and turn on notebook access. "
                       "The table at the top says where to find each one for your provider.")
import llm_client
if not hasattr(llm_client.LLMClient, "embed"):   # an older (Day 1) llm_client.py was already in this runtime
    fetch("llm_client.py", force=True)
    importlib.reload(llm_client)
from llm_client import LLMClient, LLMConfig, CALL_LOG, summarize_calls
logging.basicConfig(level=logging.WARNING, format="%(message)s", force=True)
logging.getLogger("llm_client").setLevel(logging.INFO)          # one JSON log line per call
llm = LLMClient(LLMConfig(max_retries=5, max_delay_s=30))       # free tiers: wait up to 30 s on a 429
EMBED_MODEL = os.environ["EMBED_MODEL"]

---
## D2.1 A vector you can read · *Slides 7–8*

Three made-up dimensions: **[about GPUs, about databases, about security]**.

In [ ]:
# D2.1
V = {"A": ("CUDA kernel optimisation", np.array([9, 1, 1])),
     "B": ("GPU memory bandwidth", np.array([8, 2, 1])),
     "C": ("SQL index tuning", np.array([1, 9, 2])),
     "D": ("SQL injection attack", np.array([1, 6, 9])),
     "G": ("Long survey paragraph", np.array([12, 10, 10]))}

def cosine(a, b):
    """Dot product divided by both lengths: direction only."""
    return float(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

try:
    import matplotlib.pyplot as plt
    ax = plt.figure(figsize=(6, 6)).add_subplot(projection="3d")
    for name, (label, v) in V.items():
        ax.quiver(0, 0, 0, *v, arrow_length_ratio=0.08)
        ax.text(*v, f" {name}: {label}")
    ax.set_xlabel("about GPUs"); ax.set_ylabel("about databases"); ax.set_zlabel("about security")
    ax.set_xlim(0, 12); ax.set_ylim(0, 12); ax.set_zlim(0, 12)
    plt.show()
except ImportError:
    print("(matplotlib isn't installed here, so no drawing; Colab has it)")

print("magnitude (length): square, add, square root")
for name, (label, v) in V.items():
    print(f"  |{name}| = {np.linalg.norm(v):5.2f}   {v}  {label}")

In [ ]:
# D2.1 continued: dot product, then cosine similarity
A, B, C, D, G = (V[k][1] for k in "ABCDG")
print("dot product: multiply position by position, then add")
for x, y in (("A", "B"), ("A", "C"), ("A", "G")):
    print(f"  {x} · {y} = {V[x][1] @ V[y][1]}")
print("\ncosine similarity: the dot product divided by both lengths")
for x, y in (("A", "B"), ("A", "C"), ("C", "D"), ("A", "G")):
    print(f"  cos({x}, {y}) = {cosine(V[x][1], V[y][1]):.2f}")
print("\nA · G (128) beats A · B (75), but cos(A, G) 0.76 < cos(A, B) 0.99: the focused match wins.")

---
## D2.2 Cross product, and why we don't use it · *Slide 8*

In [ ]:
# D2.2
print("A × B =", np.cross(A, B), "(a new vector at right angles to both: 3D graphics, not search)")
real = np.array(llm.embed(["How do I speed up a CUDA kernel?", "Tips for optimising GPU kernels"]))
print(f"\ntwo real embeddings from {EMBED_MODEL}: {real.shape[1]} numbers each")
try:
    np.cross(real[0], real[1])
except ValueError as e:
    print("np.cross on them →", type(e).__name__ + ":", e)

---
## D2.3 Real embeddings, real scores · *Slide 9*

**Ask first:** which pair scores highest? Is sentence 1 closer to 2 or to 3?

In [ ]:
# D2.3: one embedding
one = np.array(llm.embed(["How do I speed up a CUDA kernel?"])[0])
print(f"{EMBED_MODEL}: {len(one)} numbers, length {np.linalg.norm(one):.3f}")
print("first eight:", np.round(one[:8], 4))

In [ ]:
# D2.3: six sentences, three of them with "kernel" in three senses
SIX = ["How do I speed up a CUDA kernel?",
       "Tips for optimising GPU kernels",
       "How the Linux kernel schedules processes",
       "Choosing a kernel for an SVM classifier",
       "My SQL query is slow; should I add an index?",
       "How do I prevent SQL injection?"]
E = np.array(llm.embed(SIX))
print(f"{len(SIX)} vectors of {E.shape[1]} numbers\n")
S = (E @ E.T) / np.outer(np.linalg.norm(E, axis=1), np.linalg.norm(E, axis=1))    # cosine, every pair
print("     " + "".join(f"{j + 1:>6}" for j in range(len(SIX))))
for i, row in enumerate(S):
    print(f"{i + 1:>3}  " + "".join(f"{x:6.2f}" for x in row) + f"   {SIX[i]}")
pairs = [(S[i, j], i + 1, j + 1) for i in range(len(SIX)) for j in range(i + 1, len(SIX))]
top = max(pairs)
print(f"\nhighest pair: {top[1]} and {top[2]} ({top[0]:.2f}); 1 vs 2: {S[0, 1]:.2f}, 1 vs 3: {S[0, 2]:.2f}")

---
## D2.4 Exact identifiers · *Slide 9*

In [ ]:
# D2.4
llama = np.array(llm.embed(["Llama-3.1-8B", "Llama-3.1-70B"]))
print(f"{llama.shape[1]} numbers each")
print(f"cos(Llama-3.1-8B, Llama-3.1-70B) = {cosine(llama[0], llama[1]):.3f}")
print("Very similar vectors, very different models: exact identifiers need keyword (sparse) search or filters.")